<a href="https://colab.research.google.com/github/Chimix001/Gemmacode/blob/main/ggfGEMMACODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading Dependcies

In [1]:
!pip install -q huggingface_hub
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 || \
 CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir --force-reinstall

In [2]:
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [3]:
!pip install -U huggingface_hub hf_xet llama-cpp-python

In [4]:
import huggingface_hub
print(huggingface_hub.__version__)

1.26.0


In [ ]:
DOWNLOADING FROM HUGGUNG FACE

In [5]:
from huggingface_hub import list_repo_files

files = list_repo_files("ggml-org/gemma-4-E4B-it-GGUF")
print(files)

['.gitattributes', '.src_sha', 'README.md', 'convert.log', 'gemma-4-E4B-it-BF16.gguf', 'gemma-4-E4B-it-Q4_0.gguf', 'gemma-4-E4B-it-Q8_0.gguf', 'mmproj-gemma-4-E4B-it-BF16.gguf', 'mmproj-gemma-4-E4B-it-Q8_0.gguf', 'mtp-gemma-4-E4B-it-BF16.gguf', 'mtp-gemma-4-E4B-it-Q4_0.gguf', 'mtp-gemma-4-E4B-it-Q8_0.gguf']


In [6]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="ggml-org/gemma-4-E4B-it-GGUF",
    filename="gemma-4-E4B-it-Q4_0.gguf",
)

print(model_path)

gemma-4-E4B-it-Q4_0.gguf: reconstructing file:   0%|          |  0.00B / 4.59GB            

gemma-4-E4B-it-Q4_0.gguf: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/models--ggml-org--gemma-4-E4B-it-GGUF/snapshots/b8093469224f83f5c38f691eb906c380e9e63114/gemma-4-E4B-it-Q4_0.gguf


In [7]:
from llama_cpp import Llama
import os

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=os.cpu_count(),
    verbose=False,
)

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


TESTING THE MODEL

In [15]:
response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "create a responsive html fintech app with less explanation for a startup"}
    ],
    max_tokens=4096,
    temperature=0,
)

print(response["choices"][0]["message"]["content"])

Here is a clean, modern, and responsive HTML/CSS template for a FinTech app landing page. It uses **Tailwind CSS** via CDN for rapid, utility-first styling, which is perfect for a startup needing speed and responsiveness without writing complex custom CSS.

### `index.html`

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>NovaFin - Future of Finance</title>
    <!-- Tailwind CSS CDN for rapid styling -->
    <script src="https://cdn.tailwindcss.com"></script>
    <!-- Custom configuration for a modern FinTech look -->
    <style>
        /* Optional: Custom font import for a professional look */
        @import url('https://fonts.googleapis.com/css/projects/fira/fonts/fira-code.css');
        body {
            font-family: 'Inter', sans-serif; /* Assuming Inter is available or using system default */
        }
    </style>
</head>
<body class="bg-gray-50 text-gray-800 antia

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
SYSTEM_PROMPT = """
You are GemmaCode, an expert AI software engineer.

Rules:

1. Write clean, production-ready code.
2. Explain your reasoning briefly.
3. Fix bugs.
4. Refactor code when requested.
5. Build complete applications.
6. Follow software engineering best practices.
7. Generate complete files whenever possible.
8. If requirements are unclear, ask concise clarifying questions.
9. Always format code using Markdown code blocks.
10. When creating projects, generate the folder structure first.
"""

# Conversation Memory
# NOTE: llama-cpp-python's create_chat_completion expects plain string content
# per message (OpenAI-style), not the list-of-dict content blocks transformers'
# processor.apply_chat_template wanted.

conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

# Keep only the latest conversations
MAX_HISTORY = 10


# RAG Placeholder

def retrieve_context(query):
    """
    Later this will search Qdrant / FAISS / ChromaDB.

    For now it returns nothing.
    """
    return ""


# Generate Response

def generate_code(prompt, max_tokens=2048, temperature=0.2):

    global conversation_history

    # Retrieve documentation (future RAG)

    context = retrieve_context(prompt)

    if context:

        prompt = f"""
Reference Documentation

{context}

User Request

{prompt}
"""

    # Save user message

    conversation_history.append({"role": "user", "content": prompt})

    # Limit conversation length (keep system prompt + most recent turns)

    if len(conversation_history) > MAX_HISTORY:

        conversation_history = (
            conversation_history[:1] +
            conversation_history[-(MAX_HISTORY - 1):]
        )

    # Generate response

    response = llm.create_chat_completion(
        messages=conversation_history,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.95,
        repeat_penalty=1.05,
    )

    text = response["choices"][0]["message"]["content"].strip()
    finish_reason = response["choices"][0]["finish_reason"]

    # Save assistant response

    conversation_history.append({"role": "assistant", "content": text})

    return text, finish_reason


# Clear Conversation

def clear_chat():

    global conversation_history

    conversation_history = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    print("Conversation cleared.")

In [14]:
TASK_PROMPTS = {

    "generate": """
You are GemmaCode.

Generate complete, production-ready code.

Always:
- Produce complete files.
- Use best practices.
- Add useful comments only.
- Explain briefly before the code.
""",

    "debug": """
You are GemmaCode.

You are debugging code.

Always:
- Find every bug.
- Explain each bug.
- Show the corrected code.
- Explain why the fix works.
""",

    "review": """
You are GemmaCode.

Review the user's code.

Look for:

- Bugs
- Security issues
- Performance problems
- Readability
- Maintainability
- Best practices

Provide suggestions before rewriting.
""",

    "refactor": """
You are GemmaCode.

Refactor the user's code.

Goals:

- Cleaner structure
- Better variable names
- Better performance
- Simpler logic
- Preserve functionality
""",

    "explain": """
You are GemmaCode.

Explain the code clearly.

Teach like a senior software engineer mentoring a junior developer.

Break the explanation into sections.
"""
}


# Task Dispatcher

def run_task(task, prompt):

    if task not in TASK_PROMPTS:
        task = "generate"

    full_prompt = f"""
{TASK_PROMPTS[task]}

User Request:

{prompt}
"""

    return generate_code(full_prompt)

In [17]:

import os
import shutil

# Folder where the model will be saved
DRIVE_SAVE_PATH = "/content/drive/MyDrive/GemmaCode/model"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# Name of your GGUF model
GGUF_FILENAME = "gemma-4-E4B-it-Q4_0.gguf"

# Copy the downloaded model to Google Drive
destination = os.path.join(DRIVE_SAVE_PATH, GGUF_FILENAME)
shutil.copy(model_path, destination)

print(f"Model saved successfully!")
print(f" Location: {destination}")

Model saved successfully!
 Location: /content/drive/MyDrive/GemmaCode/model/gemma-4-E4B-it-Q4_0.gguf


confirming the model size

In [18]:
import os

file_path = "/content/drive/MyDrive/GemmaCode/model/gemma-4-E4B-it-Q4_0.gguf"

size_bytes = os.path.getsize(file_path)
size_gb = size_bytes / (1024**3)

print(f"File size: {size_gb:.2f} GB")

File size: 4.28 GB


confirming if the model is saved

In [19]:
import os

folder = "/content/drive/MyDrive/GemmaCode/model"

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    size = os.path.getsize(path) / (1024**3)
    print(f"{file}: {size:.2f} GB")

gemma-4-E4B-it-Q4_0.gguf: 4.28 GB
